In [9]:
# 1. Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA

In [10]:
# 2. Load dataset
df = pd.read_csv('Stock_price.csv')

# 3. Explore dataset
print("Dataset Head:")
print(df.head())


Dataset Head:
        Price               Close                High                 Low  \
0      Ticker              BEL.NS              BEL.NS              BEL.NS   
1        Date                 NaN                 NaN                 NaN   
2  2012-01-02  10.757936477661133   10.93730810375916  10.666858300644368   
3  2012-01-03  11.044294357299805  11.187871559035488  10.746799361241878   
4  2012-01-04  11.253098487854004   11.35571136451063  10.950036252453096   

                 Open   Volume  
0              BEL.NS   BEL.NS  
1                 NaN      NaN  
2  10.778219630865577  1220472  
3  10.976284064319463   910008  
4  10.950036252453096  2594493  


In [11]:
print("\nDataset Information:")
df.info()



Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3205 entries, 0 to 3204
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Price   3205 non-null   object
 1   Close   3204 non-null   object
 2   High    3204 non-null   object
 3   Low     3204 non-null   object
 4   Open    3204 non-null   object
 5   Volume  3204 non-null   object
dtypes: object(6)
memory usage: 150.4+ KB


In [15]:
df.isnull().sum()

Price     0
Close     1
High      1
Low       1
Open      1
Volume    1
dtype: int64

In [16]:
df.fillna()

ValueError: Must specify a fill 'value' or 'method'.

In [12]:
# 4. Convert Date into datetime and set index
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
df.sort_index(inplace=True)

KeyError: 'Date'

In [ ]:
# 6. Calculate moving averages
df['MA50'] = df['Close'].rolling(window=50).mean()
df['MA200'] = df['Close'].rolling(window=200).mean()

plt.figure(figsize=(12, 5))
plt.plot(df['Close'], label='Closing Price')
plt.plot(df['MA50'], label='50-Day Moving Average')
plt.plot(df['MA200'], label='200-Day Moving Average')
plt.title('Stock Price with Moving Averages')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.grid(True)
plt.show()

In [13]:
# 7. Seasonality analysis
df_monthly = df['Close'].resample('ME').mean()

if len(df_monthly) >= 24:
    decomposition = seasonal_decompose(
        df_monthly, model='additive', period=12
    )
    decomposition.plot()
    plt.show()
else:
    print("Not enough data for seasonal decomposition")

TypeError: Only valid with DatetimeIndex, TimedeltaIndex or PeriodIndex, but got an instance of 'RangeIndex'

In [ ]:
# 8. Correlation between price and volume
plt.figure(figsize=(8, 5))
sns.scatterplot(x=df['Volume'], y=df['Close'])
plt.title('Closing Price vs Trading Volume')
plt.xlabel('Volume')
plt.ylabel('Closing Price')
plt.grid(True)
plt.show()

correlation = df['Close'].corr(df['Volume'])
print("Correlation:", correlation)

In [ ]:
# 9. Check stationarity using ADF test
close = df['Close'].dropna()
result = adfuller(close)

print("\nADF Statistic:", result[0])
print("p-value:", result[1])

# Difference data if not stationary
if result[1] > 0.05:
    df_diff = close.diff().dropna()
else:
    df_diff = close

# 10. Plot ACF and PACF
lags = min(40, max(1, len(df_diff)//2 - 1))

fig, axes = plt.subplots(2, 1, figsize=(12, 8))
plot_acf(df_diff, ax=axes[0], lags=lags)
plot_pacf(df_diff, ax=axes[1], lags=lags)
plt.tight_layout()
plt.show()

In [14]:
# 11. Split data into training and testing
train_size = int(len(df) * 0.8)

train = df.iloc[:train_size]
test = df.iloc[train_size:]

# 12. Build ARIMA model
model = ARIMA(train['Close'], order=(5, 1, 0))
model_fit = model.fit()

print("\nARIMA Model Summary:")
print(model_fit.summary())


ValueError: Pandas data cast to numpy dtype of object. Check input data with np.asarray(data).

In [ ]:
# 13. Predict stock prices
predictions = model_fit.forecast(steps=len(test))
predictions.index = test.index

# 14. Plot actual vs predicted prices
plt.figure(figsize=(12, 5))
plt.plot(train.index, train['Close'], label='Training Data')
plt.plot(test.index, test['Close'], label='Actual Price')
plt.plot(predictions.index, predictions, label='Predicted Price')

plt.title('Stock Price Prediction using ARIMA')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.grid(True)
plt.show()

print("\nStock price forecasting completed!")
